# ☁️ Lab 3 — Pipeline RAG avec GCP Vertex AI
### MSc IAG — Cours 17 · AIvancity · Ali Mokh

Dans ce lab, vous allez reconstruire le pipeline RAG du Lab 2, mais en utilisant
**exclusivement des services Google Cloud Platform (GCP)** :

| Composant | Lab 2 (local) | Lab 3 (GCP) |
|-----------|--------------|-------------|
| Embedding | Sentence Transformers (local) | **Vertex AI `text-embedding-004`** |
| Vector Store | FAISS (local) | **ChromaDB (dans Colab)** |
| Génération | Gemini via API key | **Gemini via Vertex AI SDK** |
| Auth | Clé API | **Google OAuth (compte GCP)** |

> **Pré-requis :** Un projet GCP avec Vertex AI activé.
> Si vous n'avez pas de projet GCP, vous pouvez en créer un gratuitement avec 300$ de crédits.

## 0️⃣ Installation

> ⏳ Exécutez une seule fois.

In [ ]:
!pip install -q \
    google-cloud-aiplatform \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-google-vertexai \
    chromadb \
    pypdf \
    tqdm

print('✅ Installation terminée')

In [ ]:
import os, json, time, warnings
import numpy as np
from tqdm import tqdm
warnings.filterwarnings('ignore')

from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.config import Settings

import vertexai
from vertexai.generative_models import GenerativeModel
from vertexai.language_models import TextEmbeddingModel

print('✅ Imports OK')

## 1️⃣ Authentification GCP dans Google Colab

Colab propose une authentification directe avec votre compte Google.
Une fois connecté, vous accédez à tous vos projets GCP.

### Étapes :
1. Exécutez la cellule ci-dessous
2. Cliquez sur le lien qui apparaît
3. Connectez-vous avec votre compte Google (le même que votre projet GCP)
4. Copiez le code d'autorisation et collez-le dans la boîte

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('✅ Authentification Google réussie')

### Configuration de votre projet GCP

Remplacez les valeurs ci-dessous par celles de votre projet GCP :
- **Project ID** : visible dans la console GCP en haut de la page
- **Location** : la région de votre projet (ex. `us-central1`, `europe-west1`)

In [ ]:
# ── Configurez votre projet ───────────────────────────────────────────────
PROJECT_ID = 'VOTRE_PROJECT_ID'   # ← remplacez par votre Project ID GCP
LOCATION   = 'us-central1'         # ← ou 'europe-west1' selon votre région

# Initialiser Vertex AI
vertexai.init(project=PROJECT_ID, location=LOCATION)

print(f'✅ Vertex AI initialisé')
print(f'   Projet  : {PROJECT_ID}')
print(f'   Région  : {LOCATION}')

In [ ]:
# Test de connexion
try:
    gemini = GenerativeModel('gemini-2.0-flash-001')
    test = gemini.generate_content('Dis bonjour en une phrase.')
    print('✅ Gemini via Vertex AI fonctionne !')
    print('🤖', test.text.strip())
except Exception as e:
    print(f'❌ Erreur : {e}')
    print('   → Vérifiez que Vertex AI API est activée dans votre projet GCP')
    print('   → Console GCP → APIs & Services → Vertex AI API → Enable')

## 2️⃣ Chargement des PDFs

Même étape qu'au Lab 2 — uploadez vos fichiers PDF.

In [ ]:
from google.colab import files

PDF_DIR = '/content/pdfs'
os.makedirs(PDF_DIR, exist_ok=True)

print('⬆️  Uploadez vos PDFs :')
uploaded = files.upload()

for fname, content in uploaded.items():
    with open(f'{PDF_DIR}/{fname}', 'wb') as f:
        f.write(content)
    print(f'  ✅ {fname}')

In [ ]:
loader = DirectoryLoader(PDF_DIR, glob='*.pdf', loader_cls=PyPDFLoader, show_progress=True)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' ', '']
)
chunks = splitter.split_documents(documents)

print(f'✅ {len(documents)} pages  →  {len(chunks)} chunks')

## 3️⃣ Embeddings avec Vertex AI `text-embedding-004`

Le modèle **`text-embedding-004`** de Google est l'un des meilleurs modèles d'embedding disponibles :
- **768 dimensions** (contre 384 pour MiniLM au Lab 2)
- Entraîné sur des milliards de textes, excellent en français
- Optimisé pour la recherche sémantique (tâche `RETRIEVAL_DOCUMENT`)

| Modèle | Dimensions | Usage |
|--------|-----------|-------|
| MiniLM (Lab 2) | 384 | Local, rapide, gratuit |
| text-embedding-004 (Lab 3) | 768 | GCP, plus précis, payant |

In [ ]:
embed_model = TextEmbeddingModel.from_pretrained('text-embedding-004')

# Test sur une phrase
test_emb = embed_model.get_embeddings(
    ['Qu\'est-ce que le RAG ?'],
    task_type='RETRIEVAL_DOCUMENT'
)
print(f'✅ Modèle chargé : text-embedding-004')
print(f'   Dimension : {len(test_emb[0].values)}')
print(f'   Exemple (5 premières valeurs) : {test_emb[0].values[:5]}')

In [ ]:
def embed_texts(texts: list, task_type: str = 'RETRIEVAL_DOCUMENT',
                batch_size: int = 5) -> list:
    """
    Embedde une liste de textes avec Vertex AI.
    On traite par petits batches pour respecter les quotas API.
    task_type : 'RETRIEVAL_DOCUMENT' pour les chunks, 'RETRIEVAL_QUERY' pour les questions
    """
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Embedding'):
        batch = texts[i:i + batch_size]
        result = embed_model.get_embeddings(batch, task_type=task_type)
        all_embeddings.extend([r.values for r in result])
        time.sleep(0.1)  # Éviter les rate limits
    return all_embeddings


print('⏳ Encodage des chunks avec Vertex AI Embeddings...')
chunk_texts = [c.page_content for c in chunks]
chunk_embeddings = embed_texts(chunk_texts)

print(f'✅ {len(chunk_embeddings)} vecteurs créés')
print(f'   Dimension : {len(chunk_embeddings[0])}')

## 4️⃣ Stockage dans ChromaDB

**ChromaDB** est une base de données vectorielle open-source qui tourne
directement dans Colab (en mémoire). Elle stocke vos vecteurs et vos chunks
ensemble, ce qui simplifie la récupération.

Différence avec FAISS :
| | FAISS | ChromaDB |
|-|-------|----------|
| Stockage | Vecteurs seuls | Vecteurs + textes + métadonnées |
| Interface | Bas niveau (indices) | Haut niveau (`.query()`) |
| Persistance | Manuelle | Intégrée |

In [ ]:
# Initialiser ChromaDB en mémoire
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))

# Supprimer la collection si elle existe déjà (utile si on relance)
try:
    chroma_client.delete_collection('rag_docs')
except:
    pass

collection = chroma_client.create_collection(
    name='rag_docs',
    metadata={'hnsw:space': 'cosine'}
)

# Ajouter les chunks avec leurs embeddings
collection.add(
    ids=[f'chunk_{i}' for i in range(len(chunks))],
    embeddings=chunk_embeddings,
    documents=chunk_texts,
    metadatas=[c.metadata for c in chunks]
)

print(f'✅ ChromaDB : {collection.count()} documents indexés')

In [ ]:
def retrieve_chroma(query: str, k: int = 5) -> list:
    """Recherche dans ChromaDB avec Vertex AI Embeddings."""
    # Embedder la question (tâche RETRIEVAL_QUERY)
    query_emb = embed_model.get_embeddings(
        [query], task_type='RETRIEVAL_QUERY'
    )[0].values

    results = collection.query(
        query_embeddings=[query_emb],
        n_results=k
    )

    return [
        {'text': doc, 'score': 1 - dist, 'source': meta.get('source', '')}
        for doc, dist, meta in zip(
            results['documents'][0],
            results['distances'][0],
            results['metadatas'][0]
        )
    ]

# Test
query_test = 'Qu\'est-ce que le RAG ?'
results = retrieve_chroma(query_test, k=3)
print(f'🔍 Requête : "{query_test}"\n')
for i, r in enumerate(results, 1):
    print(f'--- Résultat {i} (score: {r["score"]:.4f}) ---')
    print(r['text'][:300])
    print()

## 5️⃣ Génération avec Gemini via Vertex AI

On utilise maintenant **Gemini via le SDK Vertex AI** (et non via une clé API directe).
C'est la façon recommandée en entreprise : les appels passent par votre projet GCP,
les coûts sont tracés, les données restent dans votre région.

In [ ]:
def rag_vertexai(question: str, k: int = 5, verbose: bool = True) -> str:
    """Pipeline RAG complet avec GCP Vertex AI."""

    # 1. Retrieve depuis ChromaDB
    retrieved = retrieve_chroma(question, k=k)
    context = '\n\n---\n\n'.join(r['text'] for r in retrieved)

    # 2. Construire le prompt
    prompt = f"""Tu es un assistant pédagogique expert.
Réponds à la question en te basant UNIQUEMENT sur le contexte fourni.
Si la réponse n'est pas dans le contexte, dis-le clairement.
Réponds en français.

CONTEXTE :
{context}

QUESTION : {question}

RÉPONSE :"""

    # 3. Générer via Vertex AI
    response = gemini.generate_content(prompt)

    if verbose:
        print(f'📚 {len(retrieved)} chunks utilisés :')
        for i, r in enumerate(retrieved, 1):
            print(f'  [{i}] score={r["score"]:.3f} — {r["text"][:80]}...')
        print()

    return response.text


# Test
question = 'Qu\'est-ce que le RAG et comment fonctionne-t-il ?'
print('💬 Question :', question, '\n')
reponse = rag_vertexai(question)
print('💡 Réponse :')
print(reponse)

## 6️⃣ Comparaison : MiniLM vs Vertex AI Embeddings

Analysons la différence qualitative entre les deux modèles d'embedding.

> 💡 Pour cette cellule, vous avez besoin des variables du **Lab 2** chargées dans la même session.
> Si vous repartez de zéro, commentez la partie Lab 2 et ne gardez que Lab 3.

In [ ]:
# Cette cellule compare les deux approches sur les mêmes questions
# Elle suppose que index et embed_model (MiniLM) du Lab 2 sont disponibles

questions = [
    'Qu\'est-ce que le RAG ?',
    'Comment fonctionne l\'attention dans un Transformer ?',
    'Quels sont les avantages de la distillation de modèle ?',
]

for q in questions:
    print(f'\n{'━'*60}')
    print(f'❓ {q}')
    print('━'*60)

    # Vertex AI
    try:
        r_vertex = retrieve_chroma(q, k=2)
        print('\n[Vertex AI text-embedding-004]')
        for r in r_vertex:
            print(f'  score={r["score"]:.4f} → {r["text"][:120]}')
    except Exception as e:
        print(f'  (Vertex AI non disponible : {e})')

## 7️⃣ Bonus — Réponse en Streaming

Vertex AI supporte le **streaming** : les tokens de réponse s'affichent
au fur et à mesure, comme dans ChatGPT. C'est bien meilleur pour l'expérience utilisateur.

In [ ]:
def rag_stream(question: str, k: int = 5):
    """RAG avec affichage en streaming."""
    retrieved = retrieve_chroma(question, k=k)
    context = '\n\n---\n\n'.join(r['text'] for r in retrieved)

    prompt = f"""Tu es un assistant pédagogique expert.
Réponds à la question en te basant UNIQUEMENT sur le contexte fourni.
Réponds en français avec une réponse structurée.

CONTEXTE :
{context}

QUESTION : {question}

RÉPONSE :"""

    print(f'💬 {question}\n')
    print('💡 Réponse (streaming) :')
    print('─' * 50)

    # Streaming
    for chunk in gemini.generate_content(prompt, stream=True):
        print(chunk.text, end='', flush=True)
    print('\n' + '─' * 50)


# Test streaming
rag_stream('Quels sont les avantages du RAG par rapport au fine-tuning ?')

---
## ✅ Récapitulatif — Lab 2 vs Lab 3

| Aspect | Lab 2 (FAISS + MiniLM) | Lab 3 (ChromaDB + Vertex AI) |
|--------|----------------------|-----------------------------|
| **Auth** | Clé API Gemini | Compte GCP + OAuth |
| **Embedding** | SentenceTransformer local (384d) | text-embedding-004 GCP (768d) |
| **Vector store** | FAISS (RAM) | ChromaDB (RAM + persistable) |
| **Génération** | Gemini via genai SDK | Gemini via Vertex AI SDK |
| **Coût** | Gratuit (modèles locaux) | Payant (API GCP) |
| **Qualité** | Bonne | Meilleure (modèle GCP) |
| **Production** | Prototype | Entreprise |

**En production sur GCP**, vous utiliseriez en plus :
- **Vertex AI Vector Search** : base vectorielle managée et scalable (remplace ChromaDB)
- **Cloud Run** : API FastAPI déployée en serverless
- **Cloud Storage** : stocker les PDFs et l'index